In [21]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import img_to_array, load_img
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define directories
train_dir = 'data/train'  # Replace with your train directory path
# validation_dir = 'data/val'  # Replace with your validation directory path

# Define augmentation parameters
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

# Define a separate generator for validation data without augmentation
val_datagen = ImageDataGenerator(rescale=1./255, validation_split = 0.2)

# Specify which classes should be augmented
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),  # Adjust target size as needed
    batch_size=32,
    class_mode='categorical',
    shuffle=True,
    subset='training',  # Specify training subset
    classes=['akiec', 'bcc', 'bkl', 'df', 'mel', 'vasc']  # Exclude 'nv'
)
new_class = 'data/train'
new_class_datagen = ImageDataGenerator(rescale=1./255)  
new_class_generator = new_class_datagen.flow_from_directory(
    new_class,
    target_size=(224, 224),  # Adjust target size as needed
    batch_size=32,
    class_mode='categorical',
    shuffle=True,
    classes=['nv']
)
# Define data generator for the new class
new_class_generator = new_class_datagen.flow_from_directory(
    new_class,
    target_size=(224, 224),  # Adjust target size as needed
    batch_size=32,
    class_mode='categorical',  # Ensure consistent label format
    shuffle=True,
    classes=['nv']  # Specify new_class only
)

# Create combined generator class
class CombinedGenerator(tf.keras.utils.Sequence):
    def __init__(self, generator1, generator2):
        self.generator1 = generator1
        self.generator2 = generator2

    def __len__(self):
        return max(len(self.generator1), len(self.generator2))
    
    def __getitem__(self, index):
        batch1 = next(self.generator1)
        batch2 = next(self.generator2)

        # Ensure batch1 and batch2 have the same number of samples
        assert batch1[0].shape[0] == batch2[0].shape[0], "Batch sizes must be equal"

        # Concatenate batches along axis=0 (batch axis)
        X_combined = np.concatenate([batch1[0], batch2[0]], axis=0)

        # Ensure labels are in the same format (e.g., one-hot encoding)
        assert batch1[1].shape[1] == batch2[1].shape[1], "Number of classes must be the same"
        
        y_combined = np.concatenate([batch1[1], batch2[1]], axis=0)

        return X_combined, y_combined

# Create combined generator
combined_generator = CombinedGenerator(train_generator, new_class_generator)

validation_generator = val_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),  # Adjust target size as needed
    batch_size=32,
    class_mode='categorical',
    subset = 'validation',
    shuffle=False  # No need to shuffle validation data
)

# Example model architecture and training
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(6, activation='softmax')  # Adjust output units based on your classes
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])



Found 3310 images belonging to 6 classes.
Found 6705 images belonging to 1 classes.
Found 6705 images belonging to 1 classes.
Found 2000 images belonging to 7 classes.


In [22]:
model.fit(
    combined_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=20,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size
)

AssertionError: Number of classes must be the same

In [ ]:
Y_pred = model.predict(validation_generator)
y_pred = np.argmax(Y_pred, axis=1)

# Get true labels
y_true = validation_generator.classes

# Compute confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.xticks(ticks=np.arange(len(validation_generator.class_indices)), labels=validation_generator.class_indices, rotation=45)
plt.yticks(ticks=np.arange(len(validation_generator.class_indices)), labels=validation_generator.class_indices, rotation=0)
plt.show()

# Print classification report
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=validation_generator.class_indices.keys()))


ValueError: Asked to retrieve element 0, but the Sequence has length 0

In [ ]:
plt.figure(figsize=(15,15))
for i in range(7):
    plt.subplot(4,2,i+1)
    true_positives = conf_matrix[i,i]
    true_negatives = np.sum(conf_matrix) - np.sum(conf_matrix[i,:]) - np.sum(conf_matrix[:,i]) + true_positives
    false_positives = conf_matrix[i,:].sum() - true_positives
    false_negatives = conf_matrix[:,i].sum() - true_positives
    mat = [[true_positives,true_negatives],[false_positives,false_negatives]]
    sns.heatmap(mat, annot=True, fmt='d', cmap='Blues', xticklabels=['Positive','Negative'], yticklabels=['True','False'])
plt.tight_layout()
plt.show()